# 介紹
使用pyautougi完成blitz的遊戲

In [ ]:
import pyautogui as pyg
pyg.sleep(2)
basex=65
basex=470
gap=40

In [3]:
import sys
import subprocess

# 自動抓取當前 Notebook 正在使用的 Python 精確路徑進行安裝
subprocess.run([sys.executable, "-m", "pip", "install", "opencv-python"])

CompletedProcess(args=['c:\\Users\\88697\\miniconda3\\envs\\iem_python\\python.exe', '-m', 'pip', 'install', 'opencv-python'], returncode=0)

In [12]:
"""
停止方式
  按 q，或把滑鼠移到螢幕左上角（pyautogui 的安全機制）
"""

from collections import Counter
import ctypes
import sys
import time
import cv2
import numpy as np
import pyautogui
from PIL import ImageGrab

# 讓 pyautogui 與截圖使用相同的實體像素座標（高 DPI 螢幕必備）
try:
    ctypes.windll.shcore.SetProcessDpiAwareness(1)
except Exception:
    pass


# =========================
# 棋盤位置（螢幕座標）
# =========================
LEFT, TOP = 655,773
RIGHT, BOTTOM = 1219, 1342
WIDTH, HEIGHT = RIGHT - LEFT, BOTTOM - TOP
N = 8

CW, CH = WIDTH / N, HEIGHT / N

# =========================
# 寶石定位
# =========================
# basex, basey：左上角寶石的中心點（螢幕座標）
basex = int(LEFT + CW / 2)
basey = int(TOP + CH / 2)

# gap：每顆寶石的間距（可給單一數值，或 (水平, 垂直)）
gap = (CW, CH)

def gem_pos(r, c):
    """回傳第 r 列（row）、第 c 行（column）那顆寶石的中心點 (x, y)。"""
    gap_x, gap_y = gap if isinstance(gap, (tuple, list)) else (gap, gap)
    return int(basex + c * gap_x), int(basey + r * gap_y)

# =========================
# 時間與流程設定
# =========================
ROUND_TIME = 65       # Bejeweled Blitz 一局約 60 秒，留一點緩衝
START_DELAY = 5       # 按下執行後，等幾秒才開始動作
STILL_LIMIT = 8          # 連續幾次找不到步可走，就判定遊戲結束
SETTLE_TIMEOUT = 0.25 # 等棋盤穩定的最長時間（秒）
VERBOSE = True           # 是否印出每次辨識到的棋盤

# 不列入消除判斷的格子
IGNORE = {"empty", "unknown"}
SYMBOL = {
    "red": "R", "orange": "O", "yellow": "Y", "green": "G",
    "blue": "B", "purple": "P", "white": "W",
    "empty": ".", "unknown": "?",
}

# =========================
# 截圖
# =========================
def capture():
    img = ImageGrab.grab(
        bbox=(LEFT, TOP, RIGHT, BOTTOM), all_screens=True
    )
    return cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)

def stop_pressed():
    """按 Q 停止（不需額外套件）。"""
    return bool(ctypes.windll.user32.GetAsyncKeyState(ord("Q")) & 0x8000)

# =========================
# 顏色辨識
# =========================
# 門檻值：跑過 calibrate / pick 之後，可依印出的數值微調
MIN_VALUE = 55      # 亮度低於此 → 空格（背景）
MIN_SAT = 45        # 飽和度低於此 → 白色（沒有明顯色彩）
SAMPLE_RADIUS = 8   # 每格中心取樣 17x17，降低反光與邊緣干擾
# 黃色、橘色使用下方實際 RGB 樣本做正規化色彩距離比對。
# 兩者色相接近，不能只依賴 HSV 色相或 G/R 比例。
# 色相（OpenCV 的 H 是 0~179）分界
HUE_RED_END = 8
HUE_ORANGE_PURE = 20   # H < 20 → 橘色
HUE_YELLOW_START = 20
HUE_YELLOW_END = 38    # H 20~38 → 再依 G/R 比例分黃、橘
HUE_GREEN_END = 85
HUE_BLUE_END = 135     # 純藍約 H=120，紫約 140 以上
# 取樣方式：kmeans 對反光、陰影、雜訊最穩（測試見 color_methods.py）
COLOR_METHOD = "kmeans"     # 可改成 "median"（較快、但遇到反光較易誤判）
def hsv_median(patch):
    """回傳一小塊畫面（BGR）的 HSV 中位數。"""
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    return tuple(int(x) for x in np.median(hsv.reshape(-1, 3), axis=0))

def bgr_median(patch):
    """回傳一小塊畫面（BGR）的 BGR 中位數。"""
    return tuple(int(x) for x in np.median(patch.reshape(-1, 3), axis=0))

def kmeans_color(patch, k=3):
    """用 K-means 把格子分成 k 群，取最大那群當作寶石本身的顏色。
    這樣反光、邊緣、背景只會落在其他小群，不會拉歪顏色。
    回傳 (h, s, v, r, g)。
    """
    data = patch.reshape(-1, 3).astype(np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0)
    _, labels, centers = cv2.kmeans(
        data, k, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
    biggest = int(np.argmax(np.bincount(labels.ravel(), minlength=k)))
    pixel = centers[biggest].reshape(1, 1, 3).astype(np.uint8)
    h, s, v = (int(x) for x in cv2.cvtColor(pixel, cv2.COLOR_BGR2HSV)[0, 0])
    b, g, r = (int(x) for x in pixel[0, 0])
    return h, s, v, r, g, b
# 依你提供的遊戲截圖校正出的主要色相中心（OpenCV H：0~179）
# 紅色跨越 0/179，因此使用圓形色相距離計算。
HUE_CENTERS = {
    "red": 176,
    "orange": 13,
    "yellow": 29,
    "green": 64,
    "blue": 106,
    "purple": 150,
}

def hue_distance(a, b):
    """計算 OpenCV 色相環上的最短距離。"""
    return min((int(a) - int(b)) % 180, (int(b) - int(a)) % 180)

def sample_color(patch):
    """直接從原始 BGR 像素取中位數，避免先投票色相造成黃橘偏色。
    回傳 (h, s, v, r, g, b)。只使用有一定彩度、亮度的像素，
    排除棋盤背景與寶石上的白色反光。
    """
    if patch is None or patch.size == 0:
        return 0, 0, 0, 0, 0, 0
    hsv = cv2.cvtColor(patch, cv2.COLOR_BGR2HSV)
    hsv_pixels = hsv.reshape(-1, 3)
    bgr_pixels = patch.reshape(-1, 3)
    valid = (hsv_pixels[:, 2] >= MIN_VALUE) & (hsv_pixels[:, 1] >= 55)
    chromatic_hsv = hsv_pixels[valid]
    chromatic_bgr = bgr_pixels[valid]
    # 沒有足夠彩色像素時，視為白色或背景。
    if len(chromatic_hsv) < max(4, int(len(hsv_pixels) * 0.08)):
        bright = hsv_pixels[hsv_pixels[:, 2] >= MIN_VALUE]
        if len(bright) == 0:
            return 0, 0, 0, 0, 0, 0
        med_s = int(np.median(bright[:, 1]))
        med_v = int(np.median(bright[:, 2]))
        if med_s < MIN_SAT:
            return 0, med_s, med_v, 255, 255, 255
        return 0, med_s, med_v, 0, 0, 0
    # HSV 與 RGB 都從同一批原始彩色像素取中位數，保留真正的 G/R、B/R 比例。
    h, s, v = (int(x) for x in np.median(chromatic_hsv, axis=0))
    b, g, r = (int(x) for x in np.median(chromatic_bgr, axis=0))
    return h, s, v, r, g, b
# 使用者提供的實際 RGB 校正值（注意：順序是 R、G、B）
YELLOW_RGB = (253, 248, 38)
ORANGE_RGB = (253, 253, 136)
def normalized_rgb_distance(rgb, reference):
    """比較色彩比例，降低整體亮度變化造成的影響。"""
    a = np.asarray(rgb, dtype=np.float32)
    b = np.asarray(reference, dtype=np.float32)
    a = a / max(float(a.sum()), 1.0)
    b = b / max(float(b.sum()), 1.0)
    return float(np.linalg.norm(a - b))

def identify(patch):
    """使用實際 RGB 樣本辨識黃橘；其他顏色沿用 HSV 色相中心。"""
    h, s, v, r, g, b = sample_color(patch)
    if v < MIN_VALUE:
        return "empty"
    if s < MIN_SAT:
        return "white"
    # 這款遊戲的黃、橘色相非常接近，因此直接比對使用者提供的 RGB 樣本。
    # 以正規化 RGB 比例比較，可容忍整體亮度稍微變亮或變暗。
    if 15 <= h <= 45:
        measured_rgb = (r, g, b)
        yellow_distance = normalized_rgb_distance(measured_rgb, YELLOW_RGB)
        orange_distance = normalized_rgb_distance(measured_rgb, ORANGE_RGB)
        return "yellow" if yellow_distance <= orange_distance else "orange"
    distances = {name: hue_distance(h, center)
                 for name, center in HUE_CENTERS.items()}
    color = min(distances, key=distances.get)
    if distances[color] > 18:
        return "unknown"
    return color

def read_board(img):
    """把截圖轉成 8x8 的顏色盤面。"""
    board = []
    for r in range(N):
        row = []
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            row.append(identify(patch))
        board.append(row)
    return board

def print_board(board):
    for row in board:
        print("  " + " ".join(SYMBOL.get(c, "?") for c in row))

def annotate(img, board):
    """把辨識結果畫在截圖上（放大 3 倍），方便肉眼檢查。"""
    out = cv2.resize(img, None, fx=3, fy=3, interpolation=cv2.INTER_NEAREST)
    for i in range(N + 1):
        x = int(i * CW * 3)
        y = int(i * CH * 3)
        cv2.line(out, (x, 0), (x, out.shape[0] - 1), (0, 255, 0), 1)
        cv2.line(out, (0, y), (out.shape[1] - 1, y), (0, 255, 0), 1)
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW * 3) - 7
            y = int((r + .5) * CH * 3) + 7
            cv2.putText(out, SYMBOL.get(board[r][c], "?"), (x, y),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2,
                        cv2.LINE_AA)
    return out

# =========================
# 消除判斷
# =========================
def find_matches(board):
    """回傳所有會被消除的格子座標集合 {(r, c), ...}。"""
    found = set()
    # 水平
    for r in range(N):
        c = 0
        while c < N:
            color = board[r][c]
            end = c
            while end < N and board[r][end] == color:
                end += 1
            if color not in IGNORE and end - c >= 3:
                found.update((r, x) for x in range(c, end))
            c = end
    # 垂直
    for c in range(N):
        r = 0
        while r < N:
            color = board[r][c]
            end = r
            while end < N and board[end][c] == color:
                end += 1
            if color not in IGNORE and end - r >= 3:
                found.update((y, c) for y in range(r, end))
            r = end
    return found

# 舊名稱，方便沿用
matches = find_matches

def move_key(a, b):
    """將交換位置標準化，避免 A<->B 與 B<->A 被當成不同步驟。"""
    return tuple(sorted((tuple(a), tuple(b))))

def best_move(board, excluded_moves=None):
    """回傳 (a, b, score)；排除本盤面已嘗試過的交換位置。"""
    if excluded_moves is None:
        excluded_moves = set()
    best = None
    best_score = 0
    for r in range(N):
        for c in range(N):
            for dr, dc in ((0, 1), (1, 0)):
                r2, c2 = r + dr, c + dc
                if r2 >= N or c2 >= N:
                    continue
                a, b = (r, c), (r2, c2)
                if move_key(a, b) in excluded_moves:
                    continue
                color = board[r][c]
                other = board[r2][c2]
                if color in IGNORE or other in IGNORE or color == other:
                    continue
                test = [row[:] for row in board]
                test[r][c], test[r2][c2] = other, color
                found = find_matches(test)
                # 交換的位置必須真的被消掉，才算是有效步
                if a not in found and b not in found:
                    continue
                score = len(found)
                if score > best_score:
                    best_score = score
                    best = (a, b, score)
    return best

def board_is_empty(board):
    """整個盤面幾乎都是空格 → 遊戲結束（或已離開遊戲）。"""
    empty = sum(cell == "empty" for row in board for cell in row)
    return empty >= N * N * 0.7

def signature(board):
    return tuple(tuple(row) for row in board)

# =========================
# 滑鼠交換
# =========================
def swap(a, b):
    x1, y1 = gem_pos(*a)
    x2, y2 = gem_pos(*b)
    pyautogui.moveTo(x1, y1)
    pyautogui.dragTo(x2, y2, duration=0.12, button="left")

def wait_stable(timeout=SETTLE_TIMEOUT):
    """等畫面連續兩次一樣，代表掉落動畫結束。"""
    deadline = time.time() + timeout
    last = None
    same = 0
    board = None
    while time.time() < deadline:
        board = read_board(capture())
        sig = signature(board)
        if sig == last:
            same += 1
            if same >= 2:
                return board
        else:
            same = 0
            last = sig
        time.sleep(0.06)
    return board

# =========================
# 校正模式
# =========================
def calibrate():
    img = capture()
    board = read_board(img)
    print(f"棋盤範圍：x {LEFT}{RIGHT}, y {TOP}{BOTTOM}（{WIDTH}x{HEIGHT}）")
    print("辨識結果（R 紅 O 橙 Y 黃 G 綠 B 藍 P 紫 W 白 . 空）：")
    print_board(board)
    print("\n各格中心 HSV / RGB（黃橘可用 g/r 比例核對）：")
    for r in range(N):
        for c in range(N):
            x = int((c + .5) * CW)
            y = int((r + .5) * CH)
            patch = img[y - SAMPLE_RADIUS:y + SAMPLE_RADIUS + 1,
                        x - SAMPLE_RADIUS:x + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            print(f"  ({r},{c}) -> {board[r][c]:7s} "
                  f"h={h:3d} s={s:3d} v={v:3d} "
                  f"rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")
            
    counts = Counter(cell for row in board for cell in row)
    print("\n顏色統計：" + "、".join(
        f"{name} x{n}" for name, n in counts.most_common()))

    if counts.most_common(1)[0][1] >= N * N * 0.9:
        print("！將近全部格子都是同一種顏色，可能畫面不在遊戲上，"
              "或棋盤座標 / 門檻需要調整")

    path = "board_debug.png"
    ok, buf = cv2.imencode(".png", annotate(img, board))
    if ok:
        buf.tofile(path)      # 用 tofile 才支援中文路徑
        print(f"\n標註圖已存成 {path}"
              "（綠線是格子，紅字是辨識結果，可用來核對）")

# =========================
# 單點取樣模式
# =========================
def pick():
    """滑鼠指到哪裡，就即時顯示那一點的顏色，用來校正門檻。"""
    print("把滑鼠移到寶石上，會即時顯示辨識結果（按 q 結束）")
    last = None
    while not stop_pressed():
        x, y = pyautogui.position()

        if LEFT <= x < RIGHT and TOP <= y < BOTTOM:
            rx, ry = x - LEFT, y - TOP
            img = capture()
            patch = img[max(ry - SAMPLE_RADIUS, 0):ry + SAMPLE_RADIUS + 1,
                        max(rx - SAMPLE_RADIUS, 0):rx + SAMPLE_RADIUS + 1]
            h, s, v, rr, g, b = sample_color(patch)
            text = (f"({x},{y}) 第 {int(ry // CH)} 列 第 {int(rx // CW)} 行"
                    f" -> {identify(patch):7s} h={h:3d} s={s:3d} v={v:3d}"
                    f" rgb=({rr:3d},{g:3d},{b:3d}) g/r={g / max(rr, 1):.2f}")
        else:
            text = f"({x},{y}) 不在棋盤範圍內"

        if text != last:
            print(text)
            last = text

        time.sleep(0.15)

# =========================
# 邏輯測試（不需要遊戲畫面）
# =========================
def selftest():
    import random

    rng = random.Random(20260101)
    colors = ["red", "green", "blue", "yellow", "purple", "orange"]
    # 固定亂數種子，找一個「沒有現成三連、但存在有效步」的盤面
    base = None
    for _ in range(5000):
        cand = [[rng.choice(colors) for _ in range(N)] for _ in range(N)]
        if not find_matches(cand) and best_move(cand) is not None:
            base = cand
            break
    assert base is not None, "找不到合適的測試盤面"
    # 橫向三連要被找到
    lined = [row[:] for row in base]
    lined[0][0] = lined[0][1] = lined[0][2] = "red"
    found = find_matches(lined)
    assert {(0, 0), (0, 1), (0, 2)} <= found, "橫向三連沒抓到"
    # 縱向三連要被找到
    lined = [row[:] for row in base]
    lined[0][4] = lined[1][4] = lined[2][4] = "blue"
    found = find_matches(lined)
    assert {(0, 4), (1, 4), (2, 4)} <= found, "縱向三連沒抓到"
    # 一定找得到有效步，而且該步真的能消
    move = best_move(base)
    assert move is not None, "找不到有效步"
    a, b, score = move
    test = [row[:] for row in base]
    test[a[0]][a[1]], test[b[0]][b[1]] = test[b[0]][b[1]], test[a[0]][a[1]]
    found = find_matches(test)
    assert a in found or b in found, "選到的步不會消除"
    assert score == len(found), "分數與實際消除數不一致"
    # 空格過多 → 視為遊戲結束
    empty = [["empty"] * N for _ in range(N)]
    assert board_is_empty(empty)
    # 顏色辨識（BGR 由 RGB 轉換）
    def patch_of(rgb):
        return np.full((9, 9, 3), tuple(reversed(rgb)), dtype=np.uint8)
    cases = [
        ((253, 248, 38), "yellow"),   # 使用者提供的黃色 RGB
        ((220, 216, 32), "yellow"),   # 黃色稍暗
        ((255, 250, 65), "yellow"),   # 黃色稍亮
        ((253, 253, 136), "orange"),  # 使用者提供的橘色 RGB
        ((220, 220, 118), "orange"),  # 橘色稍暗
        ((255, 255, 150), "orange"),  # 橘色稍亮
        ((255, 190, 0), "orange"),
        ((255, 0, 0), "red"),
        ((0, 200, 0), "green"),
        ((0, 0, 255), "blue"),
        ((200, 0, 200), "purple"),
        ((255, 255, 255), "white"),
        ((20, 20, 20), "empty"),
    ]
    for rgb, want in cases:
        got = identify(patch_of(rgb))
        assert got == want, f"rgb={rgb} 應為 {want}，卻判成 {got}"
    print(f"selftest OK（範例最佳步：{a} <-> {b}，可消 {score} 顆）")
# =========================
# 主流程
# =========================
def main():
    print("寶石方塊 自動消除")
    print(f"棋盤範圍：x {LEFT}{RIGHT}, y {TOP}{BOTTOM}")
    print(f"{START_DELAY} 秒後開始，請先切換到遊戲畫面")
    print("按 q，或把滑鼠移到左上角可停止")
    time.sleep(START_DELAY)
    start = time.time()
    moves = 0
    still = 0
    last_board = None
    attempted_signature = None
    attempted_moves = set()
    while True:
        if stop_pressed():
            print("手動停止")
            break
        elapsed = time.time() - start
        if elapsed >= ROUND_TIME:
            print(f"時間到（{elapsed:.1f} 秒），停止")
            break

        board = read_board(capture())
        if board_is_empty(board):
            print("盤面幾乎空了，遊戲可能已結束")
            break
        current_signature = signature(board)
        if current_signature != attempted_signature:
            attempted_signature = current_signature
            attempted_moves.clear()
            still = 0
        if VERBOSE and board != last_board:
            print_board(board)
            last_board = [row[:] for row in board]
        move = best_move(board, attempted_moves)
        if move is None:
            still += 1
            if attempted_moves:
                print("這個盤面剩下的有效位置都已經嘗試過，不重複操作，等待棋盤變化。")
            else:
                print(f"找不到可消除的組合（{still}/{STILL_LIMIT}）")

            if still >= STILL_LIMIT:
                print("連續找不到新步可走，判定遊戲可能結束")
                break
            time.sleep(0.15)
            continue

        still = 0
        a, b, score = move
        attempted_moves.add(move_key(a, b))
        print(f"[{elapsed:5.1f}s] 交換 {a} <-> {b}，預計消除 {score} 顆")
        swap(a, b)
        moves += 1
        wait_stable()
    print(f"結束：共交換 {moves} 次，耗時 {time.time() - start:.1f} 秒")

if __name__ == "__main__":
    mode = sys.argv[1] if len(sys.argv) > 1 else "run"
    if mode == "calibrate":
        calibrate()
    elif mode == "pick":
        pick()
    elif mode == "selftest":
        selftest()
    else:
        main()

寶石方塊 自動消除
棋盤範圍：x 6551219, y 7731342
5 秒後開始，請先切換到遊戲畫面
按 q，或把滑鼠移到左上角可停止
  . W R Y B R R R
  O G Y O G Y P W
  R B R P W G P .
  P W R G O B B O
  P P G O G G B P
  P P B W W G P P
  P R R O B R P R
  R R R B R R R R
[  0.0s] 交換 (3, 3) <-> (4, 3)，預計消除 18 顆
  . W R Y B R R R
  O G Y O G Y P W
  R B R P W G P .
  P W R G O B B O
  P P G O O W B P
  P P B W W G P P
  P R R O B R P R
  R R R B R R R R
[  1.0s] 交換 (0, 2) <-> (1, 2)，預計消除 17 顆
  . W R Y B R R R
  O G Y O G Y P W
  R B R P W G P .
  P W R G O B B O
  P P G O Y W B P
  P P B W W G P P
  P R R O B R P R
  R R R B R R R R
[  1.8s] 交換 (0, 2) <-> (1, 2)，預計消除 17 顆
  . W B Y B R R R
  O G Y O G Y P W
  R B R P W G P .
  P W R G O B B O
  P P G O O W B P
  P P B W W G P P
  P R R O B R P R
  R R R B R R R R
[  2.7s] 交換 (4, 5) <-> (5, 5)，預計消除 17 顆
  . W R Y B R R R
  O G Y O G O P W
  R B R P W G P .
  P W R G O B B O
  P P G O O G B P
  P P B P P P P P
  P R R O B R P R
  R R R B R W R R
[  3.6s] 交換 (6, 5) <-> (7, 5)，預計消除 19 顆
  . W R O 